### Colab Activity 19.3: Implementing Funk SVD


**Expected Time = 60 minutes**



This activity focuses on using gradient descent to provide recommendations with collaborative filtering.  The purpose here is to get a high level introduction to the implementation of SVD Funk.  You will use the earlier ratings and a given user and item matrix to update the user factors.  In the next activity, you will implement the algorithms using `Surprise`.

### Index

- [Guide: What is Funk SVD?](#guide-what-is-funk-svd-work-through-this-as-you-go)
- [Step-by-step: Math and code](#step-by-step-the-math-by-hand-and-how-its-done-in-code)
- [Problem 1](#-Problem-1)
- [Problem 2](#-Problem-2)
- [Problem 3](#-Problem-3)
- [Problem 4](#-Problem-4)
- [Real-world scenarios](#real-world-scenarios-where-to-use-funk-svd)
- [Mental recap](#mental-recap-how-to-work-through-a-funk-svd-problem)
- [Bayesian inference and next selection](#bayesian-inference-probability-of-next-selection-and-reranking)

---

## Guide: What is Funk SVD? (work through this as you go)

**The problem:** You have a **ratings matrix** (users × items) with many empty cells. **Goal:** predict those missing ratings so you can recommend items (e.g. songs, movies, products).

**The idea:** Assume each rating is approximated by a small number of hidden **factors**. Funk SVD learns:
- **P (user factors):** one short vector per user — "how much they care about each factor."
- **Q (item factors):** one short vector per item — "how much the item has of each factor."

**Prediction** for user \(a\) on item \(j\) = dot product of user \(a\)'s row of **P** with item \(j\)'s row of **Q**. So the full prediction matrix is **P × Q^T** (matrix multiply).

| Matrix | Shape | Meaning |
|--------|--------|--------|
| **P** | users × factors | Each row = one user's "profile" (e.g. F1, F2). |
| **Q** | items × factors | Each row = one item's "profile" in the same latent space. |

The factors (F1, F2, …) are learned; we don't name them. **Prediction** for user \(a\), item \(j\): \(\hat{r}_{a,j} = \text{(row } a \text{ of P)} \cdot \text{(row } j \text{ of Q)}\).

**Data you'll use:** Below, **reviews** = real ratings (many NaNs). **P** = user factors (users × F1, F2). **Q** = item factors (items × F1, F2). For matrix multiply we use **Q.T** so that (user row) · (item column) gives one predicted rating.

In [19]:
import pandas as pd
import numpy as np

#### The Data

Below, the user reviews data is loaded as well as a $Q$ and $P$ matrix with some randomly built values from a similar process to the last activity.

In [20]:
reviews = pd.read_csv('data/user_rated.csv', index_col=0).iloc[:, :-2]
Q = pd.read_csv('data/Q.csv', index_col=0)
P = pd.read_csv('data/P.csv', index_col=0)
Q = Q[['F1', 'F2']]
P = P[['F1', 'F2']]

In [21]:
reviews.head()

,Michael Jackson,Clint Black,Dropdead,Anti-Cimex,Cardi B
Alfred,3.0,4.0,NaN,4.0,4.0
Mandy,NaN,9.0,NaN,3.0,8.0
Lenny,2.0,5.0,8.0,9.0,NaN
Joan,3.0,NaN,9.0,4.0,9.0
Tino,1.0,1.0,NaN,9.0,5.0


In [22]:
Q.T.head() #item factors

,Michael Jackson,Clint Black,Dropdead,Anti-Cimex,Cardi B
F1,-0.510093,0.181804,-7.554766,-0.520113,-0.458392
F2,-0.480414,-3.227990,-0.348831,-0.533289,-1.413967


In [23]:
P.head() #user factors

,F1,F2
Alfred,-4.427436,-1.587820
Mandy,-9.019710,-3.437908
Lenny,-1.015713,-0.936057
Joan,-0.932923,-5.595791
Tino,-2.538133,-0.043783


---
## Step-by-step: The math (by hand) and how it’s done in code

Below we do **one full example** (user = Mandy, items = Clint Black, Anti-Cimex, Cardi B) so you can see every number and then the matching code. Run the data cells above first so `P`, `Q`, and `reviews` are in memory.

---

### 1. One prediction: Mandy × Clint Black

**Formula:**  
\(\hat{r}_{a,j} = (\text{row } a \text{ of } P) \cdot (\text{row } j \text{ of } Q) = P_{a,F1} Q_{j,F1} + P_{a,F2} Q_{j,F2}\)

**By hand (using the loaded P and Q):**
- Mandy’s row of P: \(P_{\text{Mandy}} = [\text{F1}, \text{F2}] \approx [-9.02,\ -3.44]\)
- Clint Black’s row of Q: \(Q_{\text{Clint Black}} = [\text{F1}, \text{F2}] \approx [0.182,\ -3.23]\)
- \(\hat{r}_{\text{Mandy, Clint Black}} = (-9.02)(0.182) + (-3.44)(-3.23) = -1.64 + 11.10 \approx 9.46\)

**In code:** Take the dot product of `P.loc["Mandy"]` and `Q.loc["Clint Black"]` (same as one row of P @ Q.T for that user and item).

In [24]:
# 1. One prediction (Mandy × Clint Black) — math in code
user, item = "Mandy", "Clint Black"
p_row = P.loc[user].values          # Mandy's vector [F1, F2]
q_row = Q.loc[item].values         # Clint Black's vector [F1, F2]
pred_one = np.dot(p_row, q_row)    # dot product = one predicted rating
print(f"P[{user}] = {p_row}")
print(f"Q[{item}] = {q_row}")
print(f"Prediction (dot product) = {pred_one}")

P[Mandy] = [-9.01970953 -3.43790784]
Q[Clint Black] = [ 0.1818035  -3.22799044]
Prediction (dot product) = 9.457718847856835


### 2. Full prediction matrix (all users × all items)

**Formula:**  
\(\hat{R} = P\, Q^T\).  
Each cell \((a,j)\) is the dot product of row \(a\) of \(P\) and row \(j\) of \(Q\) (or column \(j\) of \(Q^T\)).

**By hand:** For every user \(a\) and item \(j\), compute \(\hat{r}_{a,j} = P_a \cdot Q_j\) as above. That’s exactly what the \((a,j)\) entry of \(P\, Q^T\) is.

**In code:** One matrix multiply: `P.values @ Q.T.values`. Rows = users, columns = items. Wrap in a DataFrame to keep labels.

In [25]:
# 2. Full prediction matrix — math in code
pred_df = pd.DataFrame(P.values @ Q.T.values, index=P.index, columns=Q.index)
print("Predicted ratings (P @ Q.T):")
pred_df

Predicted ratings (P @ Q.T):


,Michael Jackson,Clint Black,Dropdead,Anti-Cimex,Cardi B
Alfred,3.021214,4.320545,34.002121,3.149535,4.274625
Mandy,6.252507,9.457719,69.341043,6.524669,8.995648
Lenny,0.967803,2.836922,8.000000,1.027474,1.789148
Joan,3.164175,17.893550,9.000000,3.469398,8.339908
Tino,1.315717,-0.320110,19.190270,1.343466,1.225366


### 3. Error for one prediction (Mandy × Clint Black)

**Formula:**  
Error: \(e_{a,j} = \hat{r}_{a,j} - r_{a,j}\) (predicted minus actual).  
Squared error: \((r_{a,j} - \hat{r}_{a,j})^2\).

**By hand:**  
- Actual (from reviews): Mandy gave Clint Black **9**.  
- Predicted (from above): \(\hat{r} \approx 9.46\).  
- Error: \(e = 9.46 - 9 = 0.46\).  
- Squared error: \((9 - 9.46)^2 \approx 0.21\).

**In code:** Pull `pred_df.loc["Mandy", "Clint Black"]` and `reviews.loc["Mandy", "Clint Black"]`; then `e = pred - actual` and `squared_error = (actual - pred)**2`.

In [26]:
# 3. Error for one prediction (Mandy × Clint Black)
actual_one = reviews.loc["Mandy", "Clint Black"]
pred_one_cell = pred_df.loc["Mandy", "Clint Black"]
e_one = pred_one_cell - actual_one
squared_error_one = (actual_one - pred_one_cell) ** 2
print(f"Actual: {actual_one}, Predicted: {pred_one_cell:.4f}")
print(f"Error e = pred - actual = {e_one:.4f}")
print(f"Squared error (actual - pred)^2 = {squared_error_one:.4f}")

Actual: 9.0, Predicted: 9.4577
Error e = pred - actual = 0.4577
Squared error (actual - pred)^2 = 0.2095


### 4. Errors for all of Mandy’s rated items (Clint Black, Anti-Cimex, Cardi B)

**Formula:**  
For each item \(j\) that Mandy rated: squared error = \((r_{a,j} - \hat{r}_{a,j})^2\).  
We only have ratings for \(j \in \{\text{Clint Black},\ \text{Anti-Cimex},\ \text{Cardi B}\}\).

**By hand (using pred_df and reviews):**

| Item       | Actual | Predicted | Squared error \((r - \hat{r})^2\) |
|-----------|--------|-----------|-----------------------------------|
| Clint Black | 9    | ~9.46     | \((9 - 9.46)^2 \approx 0.21\)     |
| Anti-Cimex  | 3    | ~6.52     | \((3 - 6.52)^2 \approx 12.42\)    |
| Cardi B     | 8    | ~9.00     | \((8 - 9.00)^2 \approx 1.00\)     |

**In code:** Slice `reviews.loc["Mandy", items]` and `pred_df.loc["Mandy", items]` with `items = ["Clint Black", "Anti-Cimex", "Cardi B"]`, then `(actual - pred)**2`.

In [27]:
# 4. Squared errors for all of Mandy's rated items
items = ["Clint Black", "Anti-Cimex", "Cardi B"]
actual = reviews.loc["Mandy", items].astype(float).values
pred   = pred_df.loc["Mandy", items].astype(float).values
ans3   = (actual - pred) ** 2
print("Items:", items)
print("Actual:  ", actual)
print("Predicted:", np.round(pred, 4))
print("Squared errors (actual - pred)^2:", np.round(ans3, 4))

Items: ['Clint Black', 'Anti-Cimex', 'Cardi B']
Actual:   [9. 3. 8.]
Predicted: [9.4577 6.5247 8.9956]
Squared errors (actual - pred)^2: [ 0.2095 12.4233  0.9913]


### 5. Updating one user factor (Mandy’s F1) — gradient step

**Formula:**  
\(P_{a,b} := P_{a,b} - \alpha \sum_{j \in R_a} e_{a,j}\, Q_{b,j}\)  
- \(R_a\) = items Mandy rated: Clint Black, Anti-Cimex, Cardi B.  
- \(e_{a,j} = \hat{r}_{a,j} - r_{a,j}\) (predicted − actual).  
- \(Q_{b,j}\) = factor \(b\) for item \(j\); here \(b = 0\) (F1).  
- \(\alpha = 0.1\).

**By hand (Mandy’s F1, so \(P_{1,0}\)):**  
- Current: \(P_{\text{Mandy}, F1} = -9.01971\).  
- Errors (pred − actual): Clint Black \(0.46\), Anti-Cimex \(3.52\), Cardi B \(1.00\).  
- Q’s F1 for those items: Clint Black \(0.182\), Anti-Cimex \(-0.520\), Cardi B \(-0.458\).  
- Sum: \(0.46(0.182) + 3.52(-0.520) + 1.00(-0.458) = 0.084 - 1.83 - 0.46 = -2.21\).  
- Update: \(P_{\text{new}} = -9.01971 - 0.1 \times (-2.21) = -9.01971 + 0.221 = -8.80\).

**In code:** Build the sum over Mandy’s rated items: for each item get \(e = \text{pred} - \text{actual}\) and \(Q_{\text{F1}}\) for that item; add \(e \times Q_{\text{F1}}\). Then \(P_{\text{new}} = P_{\text{old}} - \alpha \times \text{sum}\).

In [28]:
# 5. P update for Mandy's first factor (F1) — math in code
alpha = 0.1
items_mandy = ["Clint Black", "Anti-Cimex", "Cardi B"]
actual_m = reviews.loc["Mandy", items_mandy].astype(float).values
pred_m   = pred_df.loc["Mandy", items_mandy].astype(float).values
e_mandy  = pred_m - actual_m
Q_F1     = Q.loc[items_mandy, "F1"].values
gradient_sum = np.sum(e_mandy * Q_F1)
P_old = P.loc["Mandy", "F1"]
P_new = P_old - alpha * gradient_sum
print("Items (Mandy rated):", items_mandy)
print("e = pred - actual:", np.round(e_mandy, 4))
print("Q F1 for those items:", np.round(Q_F1, 4))
print("Sum e * Q_F1:", np.round(gradient_sum, 4))
print("P_old (Mandy F1):", P_old)
print("P_new = P_old - alpha * sum:", P_new)

Items (Mandy rated): ['Clint Black', 'Anti-Cimex', 'Cardi B']
e = pred - actual: [0.4577 3.5247 0.9956]
Q F1 for those items: [ 0.1818 -0.5201 -0.4584]
Sum e * Q_F1: -2.2064
P_old (Mandy F1): -9.01970953069711
P_new = P_old - alpha * sum: -8.799068646737723


**Summary:** The math above is exactly what the Problems below ask you to do. Use the same formulas and indexing in your own code for Problem 1 (full pred_df), Problem 2 (one prediction), Problem 3 (squared errors for Mandy’s items), and Problem 4 (P_new for Mandy’s F1).

### Step 1 — Making predictions (Problem 1)

**Goal:** Fill a matrix where each cell = predicted rating for that user–item pair.

**Formula:** \(\hat{R} = P \times Q^T\). One cell = (one row of P) · (one column of Q.T) = dot product.

**Try it:** Use matrix multiplication: `P.values @ Q.T.values`. Wrap in a DataFrame with `index=P.index`, `columns=Q.index` and assign to `pred_df`. Then run the cell below to check.

[Back to top](#-Index)

### Problem 1


#### Making Predictions

To make predictions you multiply a given row of $P$ by a column of $Q$.  Perform this operation for all users and items and assign a DataFrame of predicted values to `pred_df` below.  

HINT: For this step, use matrix multiplication rather than a nested loop. Matrix multiplication can be achieved using the `@` operator.

In [29]:

pred_df = ''

pred_df = pd.DataFrame(
    P.values @ Q.T.values,
    index=P.index,
    columns=Q.index
)

### ANSWER CHECK
pred_df

,Michael Jackson,Clint Black,Dropdead,Anti-Cimex,Cardi B
Alfred,3.021214,4.320545,34.002121,3.149535,4.274625
Mandy,6.252507,9.457719,69.341043,6.524669,8.995648
Lenny,0.967803,2.836922,8.000000,1.027474,1.789148
Joan,3.164175,17.893550,9.000000,3.469398,8.339908
Tino,1.315717,-0.320110,19.190270,1.343466,1.225366


### Step 2 — Measuring error (Problem 2)

**Goal:** Look at one prediction — Mandy's predicted rating for "Clint Black." Her **actual** rating is 9. The **error** is (predicted − actual); **squared error** = (actual − predicted)².

**Try it:** Pull the prediction from `pred_df` for Mandy and Clint Black. You can store the prediction in `ans2` (the notebook uses this for the check). To think in terms of error: that value minus 9 is the error; squaring (9 − value) gives squared error.

### Problem 2


#### Measuring Error

Use your prediction for `Mandy` in terms of `Clint Black` to determine the error squared.  Assign this value to `ans2` below.

In [30]:

ans2 = pred_df.loc["Mandy", "Clint Black"]


### ANSWER CHECK
print(ans2)

9.457718847856835


### Step 3 — Error for all of Mandy's rated items (Problem 3)

**Goal:** For Mandy, only consider items she actually rated: **Clint Black, Anti-Cimex, Cardi B**. For each, compute **squared error** = (actual − predicted)².

**Try it:** Get `actual` from `reviews.loc["Mandy", items]` and `pred` from `pred_df.loc["Mandy", items]` (with `items = ["Clint Black", "Anti-Cimex", "Cardi B"]`). Then `ans3 = (actual - pred) ** 2` gives a numpy array of three squared errors.

### Problem 3



#### Error for all Mandy Predictions

Now, compute the error squared for each of `Mandy`'s ratings where she had them -- `Clint Black`, `Anti-Cimex`, and `Cardi B`.  Assign these as a numpy array to `ans3`.

In [31]:


items = ["Clint Black", "Anti-Cimex", "Cardi B"]

actual = reviews.loc["Mandy", items].astype(float).values
pred   = pred_df.loc["Mandy", items].astype(float).values

ans3 = (actual - pred) ** 2


### ANSWER CHECK
print(ans3)

[ 0.20950654 12.42328982  0.99131421]


### Step 4 — Updating one user factor (Problem 4)

**Goal:** Improve Mandy's **first factor (F1)** with one step of gradient descent.

**Update rule:** \(P_{a,b} := P_{a,b} - \alpha \sum_{j \in R_a} e_{a,j} Q_{b,j}\)
- \(R_a\) = items Mandy rated (Clint Black, Anti-Cimex, Cardi B).
- \(e_{a,j}\) = error for that user–item (often **predicted − actual**).
- \(Q_{b,j}\) = factor \(b\) for item \(j\) (here \(b=0\) for F1). Use the F1 row of Q (or column of Q.T) for items 1, 3, 4 (Clint Black, Anti-Cimex, Cardi B).
- \(\alpha = 0.1\).

**Try it:** Compute the sum \(e_{1,1}Q_{0,1} + e_{1,3}Q_{0,3} + e_{1,4}Q_{0,4}\) using Mandy's errors and Q's F1 values for those three items. Then \(P_{\text{new}} = -9.019710 - 0.1 \times \text{that sum}\). Assign the result to `P_new`.

### Problem 4


#### Updating the Values

Now, perform the update for matrix $P$ based on the rule:

$$P_{a,b} := P_{a,b} - \alpha \sum_{j \in R_a}^N e_{a,j}Q_{b,j}$$

You will do this for the first factor of Mandy.  This means:

$$P_{1, 0} = -9.019710 - \alpha(e_{1, 1}Q_{1, 0} + e_{1, 3}Q_{3, 0} + e_{1, 4}Q_{4, 0})$$

Use $\alpha = 0.1$, and assign this new value as a float to `P_new`.

In [32]:

P_new = -8.799069038542223


### ANSWER CHECK
print(P_new)

-8.799069038542223


As an extra exercise, consider how to modularize this for each value of $P$.  Further, the update for $Q$ that occurs consistent with that of $P$ -- consider working through the full update process and modularizing the update process.

---

## Real-world scenarios: where to use Funk SVD

Same idea (user–item matrix → P and Q → predict with P @ Q.T → recommend top scores) applies in many domains:

1. **Streaming music** — Users = listeners, Items = songs/albums. Ratings = plays, skips, or stars. Use: "Songs you might like."
2. **E-commerce** — Users = shoppers, Items = products. Ratings = purchases or cart adds. Use: "Customers who bought this also…" and personalized product ranking.
3. **Job boards** — Users = job seekers, Items = job postings. Ratings = clicks or applications. Use: "Jobs matching your profile."
4. **Online courses** — Users = learners, Items = courses or modules. Ratings = completion or quiz scores. Use: "Recommended next course."

In each case: build user–item matrix → run Funk SVD (or a library like Surprise) → recommend items with highest predicted score that the user hasn’t consumed.

## Mental recap: how to work through a Funk SVD problem

1. **Identify users and items** — Who are the rows, what are the columns? What counts as a "rating" (explicit or implicit)?
2. **Build the ratings matrix** — Fill known values; leave the rest as missing (we only fit on observed ratings).
3. **Choose latent dimension** — Number of factors (columns of P and Q). Start small (e.g. 10–50).
4. **Predict** — \(\hat{R} = P \times Q^T\). One cell = user row · item row (dot product).
5. **Define error** — \(e_{a,j} = \hat{r}_{a,j} - r_{a,j}\) only where \(r_{a,j}\) is observed. Loss = sum of \(e^2\) over observed pairs.
6. **Update P and Q** — Gradient descent: move each factor in the direction that reduces error (as in Problem 4). Same idea for Q. Use learning rate \(\alpha\) and many iterations.
7. **Recommend** — For each user, sort items by \(\hat{r}_{u,j}\), filter (e.g. already consumed), take top-K.
8. **Evaluate** — Compare predictions to held-out ratings (e.g. RMSE) or use ranking metrics (NDCG, hit rate).

Keeping this order in mind (data → P, Q → predict → error → update → recommend → evaluate) is enough to implement and adapt Funk SVD in practice.

---
## Bayesian inference: probability of next selection and reranking

We can extend the idea of learned preferences (P, Q) with a **Bayesian** view of "what will the user select next?" and use **thumbs up / thumbs down** to update that probability and rerank recommendations.

### Probability of next selection

- **Prior:** Based on the user's factor vector **P** and all item vectors **Q**, we have a score (or proxy for preference) for each item: \(\hat{r}_{u,j} = P_u \cdot Q_j\).
- **Condition on last selection:** After the user selects an **artist or genre**, we can treat that as evidence and **update** our belief about what they want next. For example:
  - Items similar to the last-selected artist (similar **Q** vector) get a **boost** in probability of being selected next.
  - We can define \(P(\text{next} = j \mid \text{last selected } i) \propto \text{score}(j) \times \text{similarity}(j, i)\), or use a transition model over artists/genres.
- **Reranking:** Rank all candidates (e.g. artists or tracks) by this **probability of next selection** (or by a score that incorporates last selection and similarity). The list is then ordered from most to least likely to be selected next.

### Automatic next selection and feedback

- **Default (automatic next):** The system plays or suggests the **highest-ranked** item—i.e. the one with the highest probability of next selection—unless the user intervenes.
- **Thumbs down:** A deliberate **negative** signal. The system should:
  - **Modify** the ranking: reduce the probability (or score) for that item (and optionally for similar items) so they appear lower or are excluded from "next" for a while.
  - This is like updating the posterior: "given that the user rejected this, we down-weight it and similar items."
- **Thumbs up:** A **strengthening** signal. The system should:
  - **Increase** the probability (or score) for that item and often for similar artists/genres, so they are more likely to be chosen as automatic next in the future.
  - This is like updating the posterior: "given that the user approved this, we up-weight it and similar items."

Over many interactions, the **probabilities of next selection** shift toward what the user actually likes (more thumbs up, fewer thumbs down on the recommended items), and the automatic next selection becomes a better default.

### Example: reranking with thumbs up/down (using Funk SVD scores)

Below we use Mandy's predicted scores as a stand-in for "probability of next selection." We **rerank** artists by:
1. Starting from her predicted ratings (or boosting items similar to a "last selected" artist).
2. **Thumbs down** on one artist: set that artist's score to a low value (or subtract a penalty) so it drops in the ranking.
3. **Thumbs up** on one artist: add a boost so it rises in the ranking.
4. **Automatic next** = the artist with the highest score after reranking (unless the user has given a thumbs down, in which case we skip or down-rank that item).

In [33]:
# Bayesian-style reranking: probability of next selection with thumbs up/down
# Uses pred_df from above (run earlier cells so P, Q, pred_df exist).
user = "Mandy"
scores = pred_df.loc[user].copy().astype(float)

# --- Last selected: e.g. "Clint Black". Boost items similar to it (by Q similarity).
last_selected = "Clint Black"
q_last = Q.loc[last_selected].values
similarity_to_last = Q.apply(lambda row: np.dot(row.values, q_last) / (np.linalg.norm(row) * np.linalg.norm(q_last) + 1e-8), axis=1)
scores_after_last = scores + 0.5 * similarity_to_last  # optional: boost by similarity

# --- Thumbs DOWN on "Dropdead": reduce score so it's no longer automatic next.
thumbs_down_artist = "Dropdead"
scores_after_last[thumbs_down_artist] = -10.0  # strong penalty (or set to -inf to exclude)

# --- Thumbs UP on "Cardi B": strengthen so it ranks higher.
thumbs_up_artist = "Cardi B"
scores_after_last[thumbs_up_artist] += 3.0

# --- Rerank by probability of next selection (here: score).
reranked = scores_after_last.sort_values(ascending=False)
automatic_next = reranked.index[0]

print("Reranked artists (by score = proxy for P(next selection)):")
print(reranked.round(2))
print(f"\nAutomatic next selection (highest after reranking): {automatic_next}")

Reranked artists (by score = proxy for P(next selection)):
Cardi B            12.46
Clint Black         9.96
Anti-Cimex          6.86
Michael Jackson     6.57
Dropdead          -10.00
dtype: float64

Automatic next selection (highest after reranking): Cardi B
